# SysSim — ARIA Tutorial (2026-05-22)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AISysSim/SysSim/blob/lexu/demo-notebook/demo/aria_tutorial.ipynb)

SysSim estimates the step time and peak memory of LLM training on hardware
you don't have, without running real computation. This notebook walks through
five demos:

1. **Models** — Qwen3-8B vs. Llama-3-8B (dense)
2. **Configs** — Batch / Seqlen / TP / PP sweeps
3. **GPU vendor** — AMD MI300X (roofline → trained predictor)
4. **Precision** — FP8 (roofline → trained predictor)
5. **Cost model** — modifying `estimate_runtime()`

**Before you run anything:** _Runtime → Change runtime type → T4 GPU_.
(If the install cell fails on T4, fall back to L4 or A100.)


In [ ]:
import torch
assert torch.cuda.is_available(), (
    "SysSim requires a GPU runtime. "
    "Runtime → Change runtime type → T4 GPU (or L4/A100), then re-run."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Install SysSim (~3-5 min on a fresh runtime)

In [ ]:
import os, subprocess

if not os.path.exists("SysSim"):
    subprocess.run(
        'git config --global url."https://github.com/".insteadOf "git@github.com:"',
        shell=True, check=True,
    )
    subprocess.run(
        "git clone -b lexu/demo-notebook --recurse-submodules "
        "https://github.com/AISysSim/SysSim.git",
        shell=True, check=True,
    )

%cd SysSim
!pip install -q -e .
import syssim
print(f"SysSim version: {syssim.__version__}")

In [ ]:
import sys
sys.path.insert(0, ".")
from demo import helpers
print("Helpers loaded:", helpers.helpers_loaded())

## §1. Models — Qwen3-8B vs. Llama-3-8B

Same hardware (H100 DGX), same parallelism (TP=2, DP=4). The simulator
is architecture-aware — GQA group count, MLP ratio, RoPE settings all
flow through.

In [ ]:
from syssim.training.spec import load_model_yaml

QWEN3 = "examples/configs/models/qwen3-8b.yaml"
LLAMA = "demo/configs/models/llama3-8b.yaml"

for path in (QWEN3, LLAMA):
    cfg = load_model_yaml(path)
    print(f"{path}:")
    print(f"  layers={cfg.num_layers}  hidden={cfg.hidden_size}  "
          f"heads={cfg.num_attention_heads} (GQA groups={cfg.num_query_groups})  "
          f"ffn={cfg.ffn_hidden_size}  vocab={cfg.vocab_size}")

In [ ]:
import syssim
import pandas as pd

HW = "examples/configs/hardware/dgx_h100.yaml"
PAR = syssim.ParallelismConfig(tp=2, dp=4)
TR = syssim.TrainingConfig(micro_batch=1, global_batch=8, dtype="bf16")

rows = []
for name, path in [("Qwen3-8B", QWEN3), ("Llama-3-8B", LLAMA)]:
    r = syssim.simulate(model=path, hardware=HW, parallelism=PAR, training=TR)
    rows.append({
        "model": name,
        "step_time_ms": round(r.step_time_ms, 2),
        "forward_ms": round(r.forward_ms, 2),
        "backward_ms": round(r.backward_ms, 2),
        "mfu": round(r.mfu, 3),
        "peak_memory_gb": round(r.peak_memory_gb, 2),
    })
pd.DataFrame(rows)

## §2. Configs — Batch / Seqlen / TP / PP

Hold model = Qwen3-8B and HW = H100 DGX fixed. Sweep one knob at a time.

In [ ]:
import matplotlib.pyplot as plt

def run_sweep(axis_label, axis_key, values):
    rows = []
    for v in values:
        par = syssim.ParallelismConfig(tp=2, dp=4)
        tr = syssim.TrainingConfig(micro_batch=1, global_batch=8, dtype="bf16")
        if axis_key == "micro_batch":
            tr = syssim.TrainingConfig(micro_batch=v, global_batch=max(8, v*4), dtype="bf16")
        elif axis_key == "tp":
            par = syssim.ParallelismConfig(tp=v, dp=8 // v)
        elif axis_key == "pp":
            par = syssim.ParallelismConfig(pp=v, dp=8 // v)
        # NOTE: seq_length sweep below requires generating temp model YAMLs
        # because seq_length lives in the model YAML, not TrainingConfig.
        r = syssim.simulate(model=QWEN3, hardware=HW, parallelism=par, training=tr)
        rows.append({axis_label: v, "step_time_ms": round(r.step_time_ms, 2),
                     "peak_memory_gb": round(r.peak_memory_gb, 2), "mfu": round(r.mfu, 3)})
    df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar([str(v) for v in df[axis_label]], df["step_time_ms"])
    ax.set_xlabel(axis_label); ax.set_ylabel("step_time_ms")
    ax.set_title(f"Sweep: {axis_label}")
    plt.show()
    return df

for label, key, vals in [
    ("micro_batch", "micro_batch", [1, 2, 4]),
    ("TP",          "tp",          [1, 2, 4]),
    ("PP",          "pp",          [1, 2, 4]),
]:
    print(f"\n=== Sweep: {label} ===")
    display(run_sweep(label, key, vals))

In [ ]:
# Seqlen sweep: generate temp model YAMLs with different seq_length values
import tempfile, yaml
from pathlib import Path

base_cfg = yaml.safe_load(open(QWEN3))
seq_rows = []
with tempfile.TemporaryDirectory() as tmp:
    for seq in (2048, 4096, 8192):
        cfg = dict(base_cfg); cfg["seq_length"] = seq
        path = Path(tmp) / f"qwen3-8b_seq{seq}.yaml"
        path.write_text(yaml.dump(cfg))
        r = syssim.simulate(model=str(path), hardware=HW,
                            parallelism=syssim.ParallelismConfig(tp=2, dp=4),
                            training=syssim.TrainingConfig(micro_batch=1, global_batch=8, dtype="bf16"))
        seq_rows.append({"seq_length": seq, "step_time_ms": round(r.step_time_ms, 2),
                         "mfu": round(r.mfu, 3)})

seq_df = pd.DataFrame(seq_rows)
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar([str(v) for v in seq_df["seq_length"]], seq_df["step_time_ms"])
ax.set_xlabel("seq_length"); ax.set_ylabel("step_time_ms"); ax.set_title("Sweep: seq_length")
plt.show()
seq_df